# Creating LLM

In [119]:
#reading the verdict from verdict.txt file
with open('verdict.txt', 'r') as file:
    verdict = file.read().strip()
print("The length of the verdict is:", len(verdict))
print(verdict[:100])

with open ('tineystories.txt', 'r') as file:
    tineystories = file.read().strip()
print("The length of Tiney Stories is: ", len(tineystories))
print(tineystories[:100])

The length of the verdict is: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g
The length of Tiney Stories is:  19447281
Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled a


In [120]:
# class SimpleTokenizer:
#     def __init__(self,text):
#         self.dictionary = 

In [121]:
# Using tiktoken library to tokenize the verdict (BPE tokenization)
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
# print(tokenizer.max_token_value+1)
encoded = tokenizer.encode(verdict)
print("The number of tokens in the verdict is:", len(encoded))
print(encoded[:10])

#decoding the token ids back to text
decoded_ids = tokenizer.decode(encoded)
print(decoded_ids[:100])

The number of tokens in the verdict is: 5145
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [122]:
# Demo Implementing Data Sampling
context_size = 4
x = encoded[:context_size]
y = encoded[1:context_size+1]
print("X",x)
print("Y",y)

X [40, 367, 2885, 1464]
Y [367, 2885, 1464, 1807]


In [123]:
for i in range(1,context_size+1,2):
    context = encoded[:i]
    target = encoded[:i+1]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode(target)}")

I ----> I H
I HAD ----> I HAD always


In [124]:
import torch 
from torch import nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, encoded_data, tokenizer, max_length, stride):
        self.input_id = []
        self.target_id = []

        for i in range(0, len(encoded_data)-max_length, stride):
            input_chunk = encoded_data[i:i+max_length]
            target_chunk = encoded_data[i+1:i+max_length+1]
            self.input_id.append(torch.tensor(input_chunk))
            self.target_id.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_id)
    
    def __getitem__(self, idx):
        return self.input_id[idx], self.target_id[idx]

In [125]:
# Demo Implementing Data Sampling
dataset = GPTDataset(encoded, tokenizer, max_length=256, stride=128)
print(dataset)

In [126]:
def create_dataloader(txt,batch_size=8, max_length=4, stride=4,shuffle=False,drop_last = True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    encoded_text = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})
    data = GPTDataset(encoded_text, tokenizer, max_length, stride)
    dataLoader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataLoader

In [127]:
dataLoader = create_dataloader(verdict,batch_size=8,max_length=4,stride=4,shuffle=False)
data_itr = iter(dataLoader)
inputs, targets = next(data_itr)
print(inputs)
print(inputs.shape,targets.shape)
print(len(dataLoader))

#Output :
# tensor([[   40,   367,  2885,  1464],
#         [ 1807,  3619,   402,   271],
#         [10899,  2138,   257,  7026],
#         [15632,   438,  2016,   257],
#         [  922,  5891,  1576,   438],
#         [  568,   340,   373,   645],
#         [ 1049,  5975,   284,   502],
#         [  284,  3285,   326,    11]])
# torch.Size([8, 4]) torch.Size([8, 4])

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4]) torch.Size([8, 4])
160


In [128]:
# Embedding the input tokens int the form of 8*4*256
token_encodingLayer = nn.Embedding(50257,256) #Embedding layer does know that Every integer in this tensor is an index. Replace it with the corresponding embedding vector
# Output : torch.Size([8, 4, 256])

In [129]:
#Creating positional encodings for the input tokens
pos_embeddingLayer = nn.Embedding(4,256)
pos_encodings = pos_embeddingLayer(torch.arange(4))
print(pos_encodings.shape,pos_encodings)

torch.Size([4, 256]) tensor([[ 0.9523,  0.1729, -1.1594,  ...,  0.0779, -0.8313,  2.3450],
        [ 0.6350, -0.4614,  1.3238,  ...,  0.6854,  0.4720, -0.4866],
        [-0.0980,  2.3715, -0.6817,  ..., -0.3085, -1.0420,  0.1053],
        [ 0.5266, -0.8988, -0.1727,  ...,  0.2385, -1.3441, -0.8230]],
       grad_fn=<EmbeddingBackward0>)


In [130]:
# Pocessing each batch
for batch_num, (inputs, targets) in enumerate(dataLoader):
    # print(f"Processing Batch {batch_num+1}")
    token_encoding = token_encodingLayer(inputs) 
    input_embeddings = token_encoding + pos_encodings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Implementing Casual Attention Class

In [131]:
class CasualSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=True):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)   #Linear transformation to get the query vector
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
        self.dropout = nn.Dropout(dropout)
        # register_buffer is similar to nn.prameter but it is not a lernable parameter, it is a constant that is saved with the model
        self.register_buffer("mask", torch.triu(torch.ones(context_length,context_length),diagonal=1))
    
    def forward(self,x):
        b, num_token, d_in = x.shape
        keys = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)

        attn_score = query @ keys.transpose(1,2)
        attn_score = attn_score.masked_fill(self.mask[:num_token,:num_token] == 1, float("-inf"))
        attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5, dim=-1)

        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ value
        return context_vec

In [132]:
context_length = 4
ca = CasualSelfAttention(d_in=256, d_out=2, context_length=context_length, dropout=0.1)
context_vec = ca(input_embeddings)
print(context_vec.shape)

print('''
This is one of the batch which has 8 sentences with 4 tokens and each token is represented by
256 dimensional vector. After applying the casual self attention we get the context vector of each token
of shape [1,2]. This is for one token, we have 4 tokens in each sentence and 8 sentences in a batch. 
So the output shape is [8,4,2].
''')

torch.Size([8, 4, 2])

This is one of the batch which has 8 sentences with 4 tokens and each token is represented by
256 dimensional vector. After applying the casual self attention we get the context vector of each token
of shape [1,2]. This is for one token, we have 4 tokens in each sentence and 8 sentences in a batch. 
So the output shape is [8,4,2].



## Wrapper class to implement Multi-head attention 

In [133]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.head = nn.ModuleList([
            CasualSelfAttention(d_in=d_in, d_out=d_out, context_length=context_length, dropout=0.1) for _ in range(num_heads)
        ])
    
    def forward(self,x):
        return torch.cat([head(x) for head in self.head], dim=1)

In [134]:
mhaw = MultiHeadAttentionWrapper(d_in=256, d_out=2, context_length=context_length, dropout=0.1, num_heads=2)
context_v = mhaw(input_embeddings)
print(context_v.shape)
# print(context_v)

torch.Size([8, 8, 2])


Since this multihead attention mechanisum works sequentially therefore the computations requried are more.  
Implementing the multihead attention such that it can compute parallely

In [135]:
# class MulticlassAttention(nn.Module):
#     def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
#         super().__init__()
#         assert (d_out % num_heads) == 0, "d_out must be divisible by num_heads"

#         self.d_out = d_out
#         self.num_heads = num_heads
#         self.head_dim = d_out // num_heads
#         self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
#         self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)   #Linear transformation to get the query vector
#         self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) #Linear transformation to get the query vector
#         self.out_proj = nn.Linear(d_out, d_out)
#         self.dropout = nn.Dropout(dropout)
#         self.register_buffer("mask", torch.triu(torch.ones(context_length,context_length),diagonal=1))

#     def forward(self,x):
#         b, num_token, d_in = x.shape
#         keys = self.W_key(x).view(b, num_token, self.num_heads, self.head_dim).transpose(1, 2)
#         query = self.W_query(x).view(b, num_token, self.num_heads, self.head_dim).transpose(1, 2)
#         value = self.W_value(x).view(b, num_token, self.num_heads, self.head_dim).transpose(1, 2)

#         attn_Score = query @ keys.transpose(2,3)
#         mask_bool = self.mask[:num_token,:num_token]
#         attn_Score = attn_Score.masked_fill(mask_bool == 1, float("-inf"))
#         attn_weights = torch.softmax(attn_Score / self.head_dim**0.5, dim=-1)
#         attn_weights = self.dropout(attn_weights)

#         context_vec = attn_weights @ value
#         context_vec = context_vec.contiguous().view(b, num_token, self.d_out)

#         context_vec = self.out_proj(context_vec)
#         print("Context Vecor Shape is: ", context_vec.shape) # Checking the shape of Context vector 
#         return context_vec

class MulticlassAttention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False
    ):
        super().__init__()

        assert d_out % num_heads == 0, \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )
        self.W_key = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )
        self.W_value = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )

        self.out_proj = nn.Linear(
            d_out, d_out
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):

        b, num_token, d_in = x.shape

        # ----------------------------------
        # Project inputs to Q, K, V
        # ----------------------------------

        query = self.W_query(x)
        keys = self.W_key(x)
        value = self.W_value(x)

        # ----------------------------------
        # Split into heads
        # [B,T,D] -> [B,T,H,HD]
        # ----------------------------------

        query = query.view(
            b, num_token,
            self.num_heads,
            self.head_dim
        )

        keys = keys.view(
            b, num_token,
            self.num_heads,
            self.head_dim
        )

        value = value.view(
            b, num_token,
            self.num_heads,
            self.head_dim
        )

        # ----------------------------------
        # Move head dimension forward
        # [B,T,H,HD] -> [B,H,T,HD]
        # ----------------------------------

        query = query.transpose(1, 2)
        keys = keys.transpose(1, 2)
        value = value.transpose(1, 2)

        # ----------------------------------
        # Attention scores
        # [B,H,T,HD] @ [B,H,HD,T]
        # -> [B,H,T,T]
        # ----------------------------------

        attn_scores = query @ keys.transpose(2, 3)

        # ----------------------------------
        # Causal mask
        # ----------------------------------

        mask = self.mask[:num_token, :num_token]

        attn_scores = attn_scores.masked_fill(
            mask.bool(),
            float("-inf")
        )

        # ----------------------------------
        # Softmax
        # ----------------------------------

        attn_weights = torch.softmax(
            attn_scores / (self.head_dim ** 0.5),
            dim=-1
        )

        attn_weights = self.dropout(
            attn_weights
        )

        # ----------------------------------
        # Weighted sum of values
        # [B,H,T,T] @ [B,H,T,HD]
        # -> [B,H,T,HD]
        # ----------------------------------

        context_vec = attn_weights @ value

        # ----------------------------------
        # Recombine heads
        # [B,H,T,HD] -> [B,T,H,HD]
        # ----------------------------------

        context_vec = context_vec.transpose(
            1, 2
        )

        # ----------------------------------
        # [B,T,H,HD] -> [B,T,D]
        # ----------------------------------

        context_vec = context_vec.contiguous().view(
            b,
            num_token,
            self.d_out
        )

        # ----------------------------------
        # Final projection
        # ----------------------------------

        context_vec = self.out_proj(
            context_vec
        )

        return context_vec

In [136]:
# Checking Multiclass shape 

### Implementing Attention Module As Per Smallest GPT-2 Architecture

In [137]:
#implementing attention module as per smallest GPT-2 architecture
# No. of attention heads = 12, input and output dimension =768 and context length = 1024

#Preparing the input for the attention module

#step 1 : Tokeninzing the input text into 1024 of context length and embedding it into 768 dimensional vector
with open('verdict.txt', 'r') as file:
    corpus = file.read().strip()

# Step 2: Creating a DataLoader for the encoded corpus which already divides the corpus into input and target of token length 1024 and converts into batch size of 8
data_loader = create_dataloader(corpus, batch_size = 8, max_length = 1024, stride = 1024, shuffle = False)
print(len(data_loader))

# Step 3: Embedding the input tokens into 768 dimensional vector and adding positional encodings
TokenEmbeddingLayer = nn.Embedding(50257,768)
PositionalEmbeddingLayer = nn.Embedding(1024,768)
positional_encodings = PositionalEmbeddingLayer(torch.arange(1024))

for batch_num, (inputs, targets) in enumerate(data_loader):
    Token_embeddings = TokenEmbeddingLayer(inputs)
    Input_embeddings = Token_embeddings + positional_encodings
# print(Input_embeddings.shape)

# Step 4: Passing the input embeddings through the multi head attention module with 12 heads and output dimension of 768
# mha = MulticlassAttention(d_in=768, d_out=768, context_length=1024, dropout=0.1, num_heads=12)
# context_vector = mha(Input_embeddings)
# print(context_vector.shape)

0


# Implementing GPT Model from Scratch

In [138]:
GPT_CONFIG_124M = {
    "vocab_size" : 50257,
    "context_length" : 1024,
    "emb_dim" : 768,
    "n_heads" : 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [139]:
# A Placeholder GPT model architecture class

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.dropout(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5 # is a small constant added to the varuance to prevent division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim)) # learnable parameter that scales the normalized output
        self.shift = nn.Parameter(torch.zeros(emb_dim)) # learnable parameter that shifts the normalized output
    
    def forward(self,x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift


'''
GELU and SwiGLU ae more complex and smooth activation functions incorporating Gaussaian and sigmoid-gated linear units respectively.
Unlike ReLU which outputs zero for any negative input, GELU allows for small non-negative output for negative values. 
'''

class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self,x):
        return self.layers(x)

#### Creating Transformer Block Component

In [140]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MulticlassAttention(d_in=cfg["emb_dim"],
                                       d_out=cfg["emb_dim"],
                                       context_length=cfg["context_length"],
                                       num_heads=cfg["n_heads"],
                                       dropout=cfg["drop_rate"],
                                       qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    
    def forward(self,x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

In [141]:
#testing the transformer block with dummy input
torch.manual_seed(123)
dummy_input = torch.rand(8, 4, 768)
trf_block = TransformerBlock(GPT_CONFIG_124M)
output = trf_block(dummy_input)
print(dummy_input.shape)
print(output.shape)

torch.Size([8, 4, 768])
torch.Size([8, 4, 768])


In [142]:
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day hold a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DummyGPTModel(GPT_CONFIG_124M).to(device)

batch = batch.to(device)
print("batch device:", batch.device)
print("model device:", next(model.parameters()).device)

logits = model(batch)
print("logits device:", logits.device)
print("On: ", device)
print(logits.shape)

batch device: cuda:0
model device: cuda:0
logits device: cuda:0
On:  cuda
torch.Size([2, 4, 50257])


In [143]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {total_params:,}")
print(f"Size of the model in GB: {total_params * 4 / (1024**2):.2f} MB")

Total number of parameters in the model: 163,009,536
Size of the model in GB: 621.83 MB


In [144]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probas, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0).to(device)

model.eval()
out = generate_text_simple(model, idx=encoded_tensor, max_new_tokens=10, context_size=GPT_CONFIG_124M["context_length"])
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am concernsKE GlHTTP Volt starve Galestruct Klaus currents


### Training the model and Implementing loss function

In [145]:
# Implementing loss function for batch
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

# Implementing Loss function for all batches
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())
            
# Splitting the dataset into training and validation set
train_ratio = 0.9
split_idx = int(len(tineystories) * train_ratio)
train_data = tineystories[:split_idx]
val_data = tineystories[split_idx:]

GPT_CONFIG_124M = {
    "vocab_size" : 50257,
    "context_length" : 256,  # Shorten the cotext length for faster training
    "emb_dim" : 256,
    "n_heads" : 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False
}

train_loader = create_dataloader(train_data, batch_size=2, max_length=GPT_CONFIG_124M["context_length"], stride=GPT_CONFIG_124M["context_length"], shuffle=True)
val_loader = create_dataloader(val_data, batch_size=2, max_length=GPT_CONFIG_124M["context_length"], stride=GPT_CONFIG_124M["context_length"], shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print("Total Number of Batches: ", len(train_loader))


cuda
Total Number of Batches:  8330


In [146]:
# Training and evaluating the model
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokernizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, 1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train Loss {train_loss:.3f},"
                      f"Val Loss {val_loss:.3f}")
    generate_and_print_sample(model, tokenizer, device, start_context)
    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader,model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model=model, idx=encoded, max_new_tokens=50, context_size=context_size)
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()
    
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0004, weight_decay=0.1)
num_epochs = 1
train_losses, val_losses, tokens_seen = train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq=5, eval_iter=5, start_context="One day, Daisy", tokernizer=tokenizer)

Ep 1 (Step 000005): Train Loss 10.309,Val Loss 10.303
Ep 1 (Step 000010): Train Loss 9.397,Val Loss 9.305
Ep 1 (Step 000015): Train Loss 8.675,Val Loss 8.599
Ep 1 (Step 000020): Train Loss 8.006,Val Loss 7.957
Ep 1 (Step 000025): Train Loss 7.502,Val Loss 7.354
Ep 1 (Step 000030): Train Loss 6.908,Val Loss 6.860
Ep 1 (Step 000035): Train Loss 6.648,Val Loss 6.511
Ep 1 (Step 000040): Train Loss 6.460,Val Loss 6.303
Ep 1 (Step 000045): Train Loss 6.255,Val Loss 6.196
Ep 1 (Step 000050): Train Loss 6.130,Val Loss 6.108
Ep 1 (Step 000055): Train Loss 6.217,Val Loss 6.025
Ep 1 (Step 000060): Train Loss 6.045,Val Loss 6.001
Ep 1 (Step 000065): Train Loss 6.114,Val Loss 5.932
Ep 1 (Step 000070): Train Loss 5.929,Val Loss 5.867
Ep 1 (Step 000075): Train Loss 5.819,Val Loss 5.812
Ep 1 (Step 000080): Train Loss 5.857,Val Loss 5.729
Ep 1 (Step 000085): Train Loss 5.792,Val Loss 5.660
Ep 1 (Step 000090): Train Loss 5.694,Val Loss 5.615
Ep 1 (Step 000095): Train Loss 5.668,Val Loss 5.540
Ep 1 (Step

#### Temperature Scaling and Top-K Sampling 

In [147]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    # ensure tensors are on the same device as the model
    model_device = next(model.parameters()).device
    idx = idx.to(model_device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_vals, _ = torch.topk(logits, top_k, dim=-1) # Outputs Top logits, Top Positions
            min_val = top_vals[:,-1]
            logits = torch.where(logits < min_val,torch.tensor(float('-inf')).to(logits.device),logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and (idx_next == eos_id).all():
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [148]:
torch.manual_seed(123)
token_ids = generate(model=model,
                     idx=text_to_token_ids("One day, Daisy",tokenizer),
                     max_new_tokens=15,
                     context_size=GPT_CONFIG_124M["context_length"],
                     top_k=25,
                     temperature=1.4        
            )
print("output text: \n ", token_ids_to_text(token_ids, tokenizer))

output text: 
  One day, Daisy's parents was eating, so Sarah started to cry but also scared and they


In [149]:
# # Saving the model
# torch.save({
#     "model_state_dict" : model.state_dict(),
#     "optimizer_state_dict" : optimizer.state_dict(),}, "model_and_optimizer.pth")

#### Importing Weights from the gpt_download.py file

In [150]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [151]:
print(settings)

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}


In [152]:
model_configs = {
 "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
 "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
 "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
 "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024})
NEW_CONFIG.update({"qkv_bias": True})
print(NEW_CONFIG)

gpt = DummyGPTModel(NEW_CONFIG)
gpt.eval()

{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': True}


DummyGPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MulticlassAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MulticlassAttention(
        (W_query): Linear(in_feat

In [153]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, ""Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))



In [154]:
import numpy as np 
def load_weights_into_gpt(gpt, params):
   gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
   gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
   
   for b in range(len(params["blocks"])):
    q_w, k_w, v_w = np.split(
    (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
    gpt.trf_blocks[b].att.W_query.weight = assign(
    gpt.trf_blocks[b].att.W_query.weight, q_w.T)
    gpt.trf_blocks[b].att.W_key.weight = assign(
    gpt.trf_blocks[b].att.W_key.weight, k_w.T)
    gpt.trf_blocks[b].att.W_value.weight = assign(
    gpt.trf_blocks[b].att.W_value.weight, v_w.T)
    q_b, k_b, v_b = np.split(
    (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
    gpt.trf_blocks[b].att.W_query.bias = assign(
    gpt.trf_blocks[b].att.W_query.bias, q_b)
    gpt.trf_blocks[b].att.W_key.bias = assign(
    gpt.trf_blocks[b].att.W_key.bias, k_b)
    gpt.trf_blocks[b].att.W_value.bias = assign(
    gpt.trf_blocks[b].att.W_value.bias, v_b)
    gpt.trf_blocks[b].att.out_proj.weight = assign(
    gpt.trf_blocks[b].att.out_proj.weight,
    params["blocks"][b]["attn"]["c_proj"]["w"].T)
    gpt.trf_blocks[b].att.out_proj.bias = assign(
    gpt.trf_blocks[b].att.out_proj.bias,
    params["blocks"][b]["attn"]["c_proj"]["b"])
    gpt.trf_blocks[b].ff.layers[0].weight = assign(
    gpt.trf_blocks[b].ff.layers[0].weight,
    params["blocks"][b]["mlp"]["c_fc"]["w"].T)
    gpt.trf_blocks[b].ff.layers[0].bias = assign(
    gpt.trf_blocks[b].ff.layers[0].bias,
    params["blocks"][b]["mlp"]["c_fc"]["b"])
    gpt.trf_blocks[b].ff.layers[2].weight = assign(
    gpt.trf_blocks[b].ff.layers[2].weight,
    params["blocks"][b]["mlp"]["c_proj"]["w"].T)
    gpt.trf_blocks[b].ff.layers[2].bias = assign(
    gpt.trf_blocks[b].ff.layers[2].bias,
    params["blocks"][b]["mlp"]["c_proj"]["b"])
    gpt.trf_blocks[b].norm1.scale = assign(
    gpt.trf_blocks[b].norm1.scale,
    params["blocks"][b]["ln_1"]["g"])
    gpt.trf_blocks[b].norm1.shift = assign(
    gpt.trf_blocks[b].norm1.shift,
    params["blocks"][b]["ln_1"]["b"])
    gpt.trf_blocks[b].norm2.scale = assign(
    gpt.trf_blocks[b].norm2.scale,
    params["blocks"][b]["ln_2"]["g"])
    gpt.trf_blocks[b].norm2.shift = assign(
    gpt.trf_blocks[b].norm2.shift,
    params["blocks"][b]["ln_2"]["b"])
gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [155]:
load_weights_into_gpt(gpt,params)
gpt.to(device)

DummyGPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MulticlassAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MulticlassAttention(
        (W_query): Linear(in_feat

In [157]:
torch.manual_seed(123)
token_ids = generate(
 model=gpt,
 idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
 max_new_tokens=25,
 context_size=NEW_CONFIG["context_length"],
 top_k=50,
 temperature=0.8
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as the eye can see. (That's because you're not going to be able to see it all.) I


### Fine Tunning the Model - Classification Fine Tunning